# Feature selection by category: Static

Finds the strongest **static HCP** features (physician attributes, census / SDoH, colocation / IDN, market events, plus the three calendar features made in Combine), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with all 28 cohorts. Reads only this category's 4 source tables and fills empty values with the Combine rules (census features and `HCP_EXPERIENCE` use the c1-c24 median).

Same method as every category notebook: the category config (cell 3) is the only difference; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4). **Part 1:** writer definition (`TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0, empty = 0), writers / non-writers data sets, mean / median analysis (`DISCRIMINATION` = \|2 x AUC - 1\|). **Part 2:** NBA 3.1 clean-up (stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model per group on all 28 cohorts, SHAP, best features. Outputs: `feature_selection_by_category/<RUN_NAME>/static/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "static"
CATEGORY_LABEL = "Static HCP: physician attributes, census / SDoH, colocation / IDN, market events, calendar"
ADD_CALENDAR   = True    # CALENDAR_MONTH, CALENDAR_YEAR, MONTHS_SINCE_KER_LAUNCH, as made in Combine

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

# Static HCP tables of Combine, without claim_comorbidity (comorb-claims notebook) and competitor_nbrx_share (sales notebook)
SOURCE_TABLES = [
    _t("physician_attributes_features", "Physician attributes", DEV + "hcp_features/physician_attributes_features", "delta", "cohort_id"),
    _t("census_features", "Census / SDoH", DEV + "hcp_features/census_features", "delta", "cohort_id"),
    _t("colocation_idn_features", "Colocation / IDN", DEV + "hcp_features/colocation_idn_features", "delta", "cohort_id"),
    _t("market_event_features", "Market events", DEV + "hcp_features/market_event_features", "delta", "cohort_id"),
]

In [0]:
%run ./00_category_feature_selection_core